In [10]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [ ]:
# train.py
import os, math, json, random, argparse, time, sys
sys.path.append(os.path.abspath("C:\\Users\\ammaa\\OneDrive\\Documents\\GTAAutoDrive")) # Replace with the actual path
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils
from tqdm import tqdm
from dataset import FramesControls, IMNET_MEAN, IMNET_STD
from model import StudentPolicy

ModuleNotFoundError: No module named 'dataset'

In [ ]:
def loss_fn(pred, target, w=(0.8,0.15,0.05)):
    w = torch.tensor(w, device=pred.device).view(1,3)
    
    # Squared error with increased focus on steering
    mse = ((pred - target)**2 * w).sum(dim=1)
    
    # Additional L1 loss term specifically for steering to prevent stagnation
    steer_l1 = 0.2 * (pred[:,0] - target[:,0]).abs()
    
    # Extra penalty for large steering errors (over 0.2)
    steer_threshold = 0.2
    large_steer_errors = torch.relu(pred[:,0] - target[:,0]).abs() - steer_threshold
    large_error_penalty = 0.3 * torch.relu(large_steer_errors)
    
    return (mse + steer_l1 + large_error_penalty).mean()

In [ ]:
def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    # number of previous frames to stack (0 => single frame)
    num_prev = max(0, int(args.prev_frames))
    in_ch = 3 * (1 + num_prev)

    # TensorBoard writer
    logdir = args.logdir if getattr(args, 'logdir', None) else os.path.join("runs", f"exp-{time.strftime('%Y%m%d-%H%M%S')}")
    writer = SummaryWriter(log_dir=logdir)
    print(f"TensorBoard logs -> {logdir}")
    global_step = 0

    ds = FramesControls(args.frames, args.labels, split_file=None,
                        train=True, input_size=args.size, use_prev=num_prev)

    # split into train/val (or pass explicit split files for environment-wise split)
    val_ratio = 0.2
    n_val = max(200, int(len(ds)*val_ratio))
    n_train = len(ds) - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))

    train_loader = DataLoader(train_ds, batch_size=args.bs, shuffle=True, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=args.bs, shuffle=False, num_workers=4, pin_memory=True)

    model = StudentPolicy(in_ch=in_ch, meta_dim=0, aux_tlight=args.aux_tlight).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    
    # Configure learning rate schedule
    if args.schedule == "cosine":
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    elif args.schedule == "step":
        # Reduce LR by 0.1 at 50% and 75% of training
        milestones = [args.epochs // 2, args.epochs * 3 // 4]
        sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=milestones, gamma=0.1)
    elif args.schedule == "none":
        sched = None
    else:
        raise ValueError(f"Unknown schedule: {args.schedule}")

    ce = nn.CrossEntropyLoss()

    best = math.inf
    for epoch in range(1, args.epochs+1):
        model.train()
        tbar = tqdm(train_loader, desc=f"epoch {epoch} train")
        tr_loss = 0.0
        for x, y, tcls in tbar:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            if args.aux_tlight:
                pred, tlogits = model(x)
                l = loss_fn(pred, y)
                l += 0.2*ce(tlogits, tcls.to(device))
            else:
                pred = model(x)
                l = loss_fn(pred, y)
            l.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tr_loss += l.item()*x.size(0)
            current_lr = opt.param_groups[0]["lr"]
            tbar.set_postfix(loss=f"{l.item():.4f}", lr=f"{current_lr:.2e}")

            # log training scalars every N steps
            global_step += 1
            if global_step % args.log_every == 0:
                writer.add_scalar("train/loss", l.item(), global_step)
                writer.add_scalar("train/lr", current_lr, global_step)
        if sched is not None:
            sched.step()

        # validate
        model.eval()
        va_loss, n = 0.0, 0
        mae_steer = 0.0
        with torch.no_grad():
            for x, y, tcls in tqdm(val_loader, desc="val"):
                x, y = x.to(device), y.to(device)
                if args.aux_tlight:
                    pred, tlogits = model(x)
                    l = loss_fn(pred, y) + 0.2*ce(tlogits, tcls.to(device))
                else:
                    pred = model(x)
                    l = loss_fn(pred, y)
                va_loss += l.item()*x.size(0)
                mae_steer += (pred[:,0].abs() - y[:,0].abs() + (pred[:,0]-y[:,0]).abs()).sum().item()*0 + (pred[:,0]-y[:,0]).abs().sum().item()
                n += x.size(0)
        va_loss /= n
        mae_steer /= n
        print(f"[epoch {epoch}] val loss: {va_loss:.4f}  steer MAE: {mae_steer:.4f}")

        # log validation scalars
        writer.add_scalar("val/loss", va_loss, epoch)
        writer.add_scalar("val/mae_steer", mae_steer, epoch)

        # optionally log a batch of validation images (current frame) every log_img_freq epochs
        if getattr(args, 'log_img_freq', 0) and (epoch % args.log_img_freq == 0):
            try:
                xb, yb, _ = next(iter(val_loader))
                # always visualize the current frame (last 3 channels)
                xb_vis = xb[:, -3:, :, :]
                mean = torch.tensor(IMNET_MEAN).view(1,3,1,1)
                std  = torch.tensor(IMNET_STD).view(1,3,1,1)
                xb_vis = xb_vis * std + mean
                xb_vis = xb_vis.clamp(0.0, 1.0)
                xb_vis = xb_vis.cpu()
                grid = vutils.make_grid(xb_vis, nrow=min(8, xb_vis.size(0)))
                writer.add_image("val/images", grid, epoch)
            except Exception as e:
                print("Image logging skipped:", e)

        writer.flush()

        if va_loss < best:
            best = va_loss
            os.makedirs("checkpoints", exist_ok=True)
            torch.save({"model": model.state_dict(), "args": vars(args)}, "checkpoints/best.pth")
            print("✓ saved checkpoints/best.pth")

    # close TensorBoard writer
    try:
        writer.close()
    except Exception:
        pass